## CLIP testing

In [1]:
from sklearn.model_selection import StratifiedShuffleSplit
import pandas as pd

from train.train import *
from configs.deterministic import *

if __name__ == "__main__":
    set_seed(CFG.SEED,CFG.DETERMINISTIC)
    df_long = pd.read_csv(os.path.join('csiro-biomass', 'train.csv'))
    df_wide = df_long.pivot(index='image_path', columns='target_name', values='target').reset_index()
    df_wide = df_wide[['image_path'] + ['Dry_Green_g','Dry_Dead_g','Dry_Clover_g','GDM_g','Dry_Total_g']]
    aux_cols = ['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm']
    df_aux = df_long[aux_cols].drop_duplicates().reset_index(drop=True)
    df_wide = df_wide.merge(df_aux, on='image_path', how='left')
    df_wide['biomass_bin'] = pd.qcut(df_wide['Dry_Total_g'], q=10, labels=False)
    df_wide['stratify_key'] = df_wide['Species'].astype(str) + "_" + df_wide['State'].astype(str)
    
    # Check for singletons! 
    # Stratified Split crashes if a group has only 1 member (can't split 1 item into 2 sets).
    # We filter out rare groups or assign them to a "misc" group for the split.
    counts = df_wide['stratify_key'].value_counts()
    singletons = counts[counts < 2].index
    
    # Fallback: If a group has only 1 sample, we treat it as part of a generic group 
    # just for the purpose of splitting, so the code doesn't crash.
    split_key = df_wide['stratify_key'].apply(lambda x: 'misc' if x in singletons else x)

    # --- C. PERFORM STRATIFIED SPLIT ---
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    
    # This returns the INDICES for the split
    train_idx, val_idx = next(splitter.split(df_wide, split_key))
    train_idx = [
    0, 1, 3, 4, 5, 6, 8, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 22,
    25, 26, 27, 28, 30, 31, 32, 33, 35, 38, 40, 41, 42, 46, 48, 49, 50, 51,
    53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 71, 72,
    73, 74, 75, 76, 77, 78, 79, 80, 81, 83, 84, 85, 86, 87, 88, 89, 90, 91,
    92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 104, 105, 106, 107, 108, 109, 111,
    112, 115, 116, 119, 120, 121, 122, 123, 124, 126, 127, 128, 130, 133, 134, 135, 136, 137,
    138, 139, 141, 142, 143, 144, 145, 146, 148, 149, 150, 152, 153, 154, 155, 157, 158, 159,
    160, 161, 163, 164, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 177, 178, 179, 180,
    183, 184, 185, 186, 187, 188, 190, 192, 193, 194, 196, 197, 199, 200, 201, 202, 204, 205,
    206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223,
    224, 225, 226, 227, 228, 229, 232, 233, 235, 236, 237, 239, 240, 241, 242, 243, 244, 246,
    247, 250, 251, 252, 253, 255, 256, 257, 258, 259, 260, 261, 263, 265, 266, 268, 269, 270,
    271, 272, 275, 276, 277, 278, 279, 281, 283, 284, 285, 287, 289, 290, 291, 293, 294, 295,
    296, 298, 300, 301, 302, 303, 304, 305, 306, 307, 309, 310, 311, 312, 314, 315, 318, 319,
    320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337,
    339, 340, 341, 342, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 355
    ]

    val_idx = [
        2, 7, 9, 18, 21, 23, 24, 29, 34, 36, 37, 39, 43, 44, 45, 47, 52, 69,
        70, 82, 103, 110, 113, 114, 117, 118, 125, 129, 131, 132, 140, 147, 151, 156, 162, 165,
        176, 181, 182, 189, 191, 195, 198, 203, 230, 231, 234, 238, 245, 248, 249, 254, 262, 264,
        267, 273, 274, 280, 282, 286, 288, 292, 297, 299, 308, 313, 316, 317, 338, 343, 354, 356
    ]

    print(val_idx)
    # Create the actual sub-dataframes
    train_df = df_wide.iloc[train_idx].reset_index(drop=True)
    val_df = df_wide.iloc[val_idx].reset_index(drop=True)

    model = train_clip(train_df, val_df)
    #Epoch 1 | Train Loss: 0.6357 | Val Loss: 6.0239 | Val Acc 1: 2.78% | Val Acc 5: 4.17%
    #Epoch 25 | Train Loss: 0.2737 | Val Loss: 3.3724 | Val Acc 1: 5.56% | Val Acc 5: 40.28%
    #Epoch 25 | Train Loss: 0.3212 | Val Loss: 3.3872 | Val Acc 1: 11.11% | Val Acc 5: 40.28%

/home/teo/kaggle/csiro/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2, 7, 9, 18, 21, 23, 24, 29, 34, 36, 37, 39, 43, 44, 45, 47, 52, 69, 70, 82, 103, 110, 113, 114, 117, 118, 125, 129, 131, 132, 140, 147, 151, 156, 162, 165, 176, 181, 182, 189, 191, 195, 198, 203, 230, 231, 234, 238, 245, 248, 249, 254, 262, 264, 267, 273, 274, 280, 282, 286, 288, 292, 297, 299, 308, 313, 316, 317, 338, 343, 354, 356]
Loading OpenCLIP model: convnext_base_w...
trainable params: 721,920 || all params: 88,943,744 || trainable%: 0.8117
Pre-computing text anchors...
Anchors computed. Starting Training...
Regenerating global text anchors for this epoch...


Epoch 1 | Train Loss: 0.6851 | Val Loss: 3.8161 | Val Acc 1: 4.17% | Val Acc 5: 25.00%
--> New Best Loss! Saving model...
Regenerating global text anchors for this epoch...


Epoch 2 | Train Loss: 0.6154 | Val Loss: 3.6701 | Val Acc 1: 4.17% | Val Acc 5: 25.00%
--> New Best Loss! Saving model...
Regenerating global text anchors for this epoch...


Epoch 3 | Train Loss: 0.5862 | Val Loss: 3.3596 | Val Acc 1: 9.72% | Val Acc 5: 33.33%
--> New Best Loss! Saving model...
Regenerating global text anchors for this epoch...


...

Epoch 13 | Train Loss: 0.4905 | Val Loss: 3.5862 | Val Acc 1: 4.17% | Val Acc 5: 43.06%
Regenerating global text anchors for this epoch...


Epoch 14 | Train Loss: 0.4804 | Val Loss: 3.9129 | Val Acc 1: 6.94% | Val Acc 5: 30.56%
Regenerating global text anchors for this epoch...


Epoch 15 | Train Loss: 0.4839 | Val Loss: 3.5132 | Val Acc 1: 8.33% | Val Acc 5: 37.50%
Regenerating global text anchors for this epoch...


Epoch 16 | Train Loss: 0.4827 | Val Loss: 3.4904 | Val Acc 1: 8.33% | Val Acc 5: 37.50%
Regenerating global text anchors for this epoch...


Epoch 17 | Train Loss: 0.4799 | Val Loss: 3.7238 | Val Acc 1: 6.94% | Val Acc 5: 30.56%
EARLY STOP (no improvement in 10 epochs)


In [3]:
from utils.utils import *
clip_state_dict = get_clean_timm_state_dict(model)

In [ ]:
torch.save(clip_state_dict, os.path.join(CFG.MODEL_DIR, f'saved_state_dict.pth'))

In [1]:
import torch, os
from train.train import *
from configs.deterministic import *
clip_state_dict = torch.load(os.path.join(CFG.MODEL_DIR, f'pretrained_backbone.pth'), map_location='cpu')


/home/teo/kaggle/csiro/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Compare pretrained backbones

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from configs.cfg import CFG
from dataset.biomass_dataset import *
from utils.augs import *
from configs.deterministic import *
from models.models import *
from train.train import *
from dataset.preprocess_data import *

if __name__ == "__main__":
    set_seed(CFG.SEED,CFG.DETERMINISTIC)
    df_wide = get_df()
    
    fold_indices = {}

    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    splitter = sgkf.split(X=df_wide, y=df_wide['biomass_bin'], groups=df_wide['group'])

    for fold_id, (train_idx, val_idx) in enumerate(splitter):
        fold_indices[fold_id] = val_idx
        print(f"Fold {fold_id} captured: {len(val_idx)} images")

    # A. Define which folds go where
    train_folds = [1, 2, 3]
    val_fold    = 4
    test_fold   = 0

    # B. Construct the final index lists
    # np.concatenate joins the arrays of indices together
    train_idx_final = np.concatenate([fold_indices[f] for f in train_folds])
    val_idx_final   = fold_indices[val_fold]
    test_idx_final  = fold_indices[test_fold]
    # C. Create the DataFrames
    train_df = df_wide.iloc[train_idx_final].reset_index(drop=True)
    val_df   = df_wide.iloc[val_idx_final].reset_index(drop=True)
    test_df  = df_wide.iloc[test_idx_final].reset_index(drop=True)

    print(f"Train Size: {len(train_df)}") # Should be roughly 60%
    print(f"Val Size:   {len(val_df)}")   # Should be roughly 20%
    print(f"Test Size:  {len(test_df)}")  # Should be roughly 20%
    # print(val_idxf3)
    # Create the actual sub-dataframes
    # train_df = df_wide.iloc[train_idxf3].reset_index(drop=True)
    # val_df = df_wide.iloc[val_idxf3].reset_index(drop=True)
    # [5267,4500,5583]
    group_name = f"Date_{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    best_r2 = train_base(train_df,val_df, 0,model_state_dict=None,group_name=group_name,test_df=test_df)

/home/teo/kaggle/csiro/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading data...
357 training images
Fold 0 captured: 76 images
Fold 1 captured: 74 images
Fold 2 captured: 90 images
Fold 3 captured: 57 images
Fold 4 captured: 60 images
Train Size: 221
Val Size:   76
Test Size:  60
Building model...
vit_small_patch16_dinov3 parameters: 21586944
Pretrained weights loaded (CPU)
Freezing backbone parameters.


wandb: Currently logged in as: butnaruteodor (butnaruteodor-universitatea-tehnic-gheorghe-asachi-din-ia-i) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


train:   0%|          | 0/221 [00:00<?, ?it/s]/home/teo/kaggle/csiro/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 01 | TrainLoss 1916.74744 | ValLoss 2275.43581 | ValR² -1.1937 (BEST)
TestLoss 1583.31745 | TestR2: -1.5675
SAVED (R²: -1.1937)


...

TestLoss 200.35144 | TestR2: 0.6750


TestLoss 190.40759 | TestR2: 0.6914


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


TestLoss 199.55300 | TestR2: 0.6764
EARLY STOP (no improvement in 15 epochs)


best_r2,▁▂▇▇████████████████████████████████████
test_loss,█▆▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test_val_r2,▁██▇████████████████████████████████████
train_loss,█▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▄▂▂▁▂▁▁▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_r2,▁▆▇▇▇▇▇▇████████████████████████████████
best_r2,0.81363
test_loss,199.553
test_val_r2,0.67643
train_loss,116.17227
val_loss,236.65167


In [1]:
import gc
torch.cuda.empty_cache()
gc.collect()
del model, train_loader, val_loader, optimizer, main_scheduler

NameError: name 'torch' is not defined

## Convert Adapters

In [3]:
import torch
import open_clip
from peft import PeftModel
from collections import OrderedDict

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Must match exactly what you used during training

# The folder where your best LoRA weights are saved
LORA_PATH = "adapters/r8" 
MODEL_NAME = "convnext_base_w" 
PRETRAINED = "laion2b_s13b_b82k_augreg"
# The output file name you will load in the other notebook
OUTPUT_FILENAME = "lora_finetuned_convnext_base_r8.pt"

# ==========================================
# 2. MERGE
# ==========================================
print(f"Loading base model: {MODEL_NAME}...")
base_model, _, _ = open_clip.create_model_and_transforms(
    MODEL_NAME, 
    pretrained=PRETRAINED
)

print(f"Loading LoRA adapters from: {LORA_PATH}...")
base_model.visual = PeftModel.from_pretrained(base_model.visual, LORA_PATH)

print("Merging LoRA weights...")
# This gives us the merged visual model (still has OpenCLIP specific names)
merged_visual_model = base_model.visual.merge_and_unload()
raw_state_dict = merged_visual_model.state_dict()
print(merged_visual_model)
# ==========================================
# 3. CLEANING (The Magic Step)
# ==========================================
print("Cleaning state dict keys...")
clean_state_dict = OrderedDict()

for key, value in raw_state_dict.items():
    # 1. Remove 'trunk.' (OpenCLIP puts everything under trunk for ConvNeXt)
    new_key = key.replace("trunk.", "")
    
    # 2. Remove 'visual.' (Just in case)
    new_key = new_key.replace("visual.", "")
    
    # 3. Remove 'module.' (If DDP was used)
    new_key = new_key.replace("module.", "")
    
    # 4. Handle Head discrepancies
    if "head.proj" in new_key:
        print(f"Skipping CLIP projection layer: {new_key}")
        continue  # Don't add this to the new dict
    
    clean_state_dict[new_key] = value

# ==========================================
# 4. SAVE
# ==========================================
print(f"Saving cleaned weights to {OUTPUT_FILENAME}...")
torch.save(clean_state_dict, OUTPUT_FILENAME)

print("Done! The file is now compatible with standard timm loading.")

Loading base model: convnext_base_w...
Loading LoRA adapters from: adapters/r8...
Merging LoRA weights...
TimmModel(
  (trunk): ConvNeXt(
    (stem): Sequential(
      (0): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((128,), eps=1e-06, elementwise_affine=True)
    )
    (stages): Sequential(
      (0): ConvNeXtStage(
        (downsample): Identity()
        (blocks): Sequential(
          (0): ConvNeXtBlock(
            (conv_dw): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
            (norm): LayerNorm((128,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=128, out_features=512, bias=True)
              (act): GELU()
              (drop1): Dropout(p=0.0, inplace=False)
              (norm): Identity()
              (fc2): Linear(in_features=512, out_features=128, bias=True)
              (drop2): Dropout(p=0.0, inplace=False)
            )
            (shortcut): Ident

## Cross Validation Training

In [2]:
import pandas as pd
import gc
from datetime import datetime
from sklearn.model_selection import StratifiedGroupKFold
from configs.cfg import CFG
from torch.amp import GradScaler
from dataset.biomass_dataset import *
from utils.augs import *
from configs.deterministic import *
from models.models import *
from train.train import *
from dataset.preprocess_data import *
from utils.utils import *

# Define your weights exactly as you listed them
WEIGHTS_MAP = {
    'Dry_Green_g': 0.1,
    'Dry_Dead_g':  0.1,
    'Dry_Clover_g': 0.1,
    'GDM_g':       0.2,
    'Dry_Total_g': 0.5
}

def calculate_weighted_score(df):
    """Helper to create the column just like you did in check_splits"""
    # Start with 0
    weighted_col = np.zeros(len(df))
    for col, w in WEIGHTS_MAP.items():
        weighted_col += df[col] * w
    return weighted_col

def find_best_seed(df, n_seeds=2000):
    """
    Iterates through random seeds to find the one that minimizes the 
    statistical difference between folds for critical targets.
    """
    
    # 1. Prepare Data for Search
    # We calculate the weighted column purely for the balancing metric
    df_search = df.copy()
    df_search['Weighted_g'] = calculate_weighted_score(df_search)
    
    # We want to balance these specific columns. 
    # We prioritize 'Dry_Clover_g' because it is sparse/weird (your Fold 4 issue).
    # We prioritize 'Weighted_g' because it represents the overall difficulty.
    targets_to_balance = ['Weighted_g', 'Dry_Clover_g', 'Dry_Dead_g']
    
    best_seed = -1
    lowest_penalty = float('inf')
    
    print(f"🔍 Searching {n_seeds} seeds for the most balanced split...")
    
    for seed in tqdm(range(n_seeds)):
        # Initialize the splitter with the current seed
        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        
        # We collect the means of our targets for each fold
        fold_means = {t: [] for t in targets_to_balance}
        
        try:
            # Iterate through the folds
            for _, val_idx in sgkf.split(df_search, df_search['biomass_bin'], groups=df_search['group']):
                val_fold = df_search.iloc[val_idx]
                
                for t in targets_to_balance:
                    fold_means[t].append(val_fold[t].mean())
            
            # --- CALCULATE PENALTY ---
            # We use Coefficient of Variation (Std Dev / Mean). 
            # This normalizes the score so 'Total_g' (big numbers) doesn't overpower 'Clover' (small numbers).
            current_penalty = 0
            for t in targets_to_balance:
                means_array = np.array(fold_means[t])
                # If global mean is 0, avoid division by zero
                global_mean = df_search[t].mean() + 1e-6 
                
                # How much do the folds deviate from each other?
                cv = np.std(means_array) / global_mean
                current_penalty += cv
            
            # If this seed is more balanced than the previous best, save it
            if current_penalty < lowest_penalty:
                lowest_penalty = current_penalty
                best_seed = seed
                
        except ValueError:
            continue

    print("-" * 50)
    print(f"✅ Best Seed Found: {best_seed}")
    print(f"📉 Penalty Score: {lowest_penalty:.4f} (Lower is better)")
    
    return best_seed

if __name__ == "__main__":
    set_seed(CFG.SEED, CFG.DETERMINISTIC)
    df_wide = get_df()
    # best_seed = find_best_seed(df_wide, 1000)
    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    splitter = sgkf.split(X=df_wide, y=df_wide['biomass_bin'], groups=df_wide['group'])

    # check_splits(splitter, df_wide)
    # Store the best R2 score from each fold
    all_fold_best_scores = []
    group_name = f"Date_{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    for fold, (tr_idx, val_idx) in enumerate(splitter):
        print('\n' + '='*70)
        print(f'   FOLD {fold+1}/{CFG.N_FOLDS}   |   {len(tr_idx)} train / {len(val_idx)} val')
        print('='*70)
        
        # print(tr_idx)
        # print(val_idx)

        tr_df  = df_wide.iloc[tr_idx].reset_index(drop=True)
        val_df = df_wide.iloc[val_idx].reset_index(drop=True)

        best_r2 = train_base(tr_df,val_df,fold,group_name = group_name)
        all_fold_best_scores.append(best_r2)

    # Cleanup
    torch.cuda.empty_cache()
    gc.collect()
    all_fold_sizes = [76,74,90,57,60]# Change by hand if folds change
    weighted_cv_score = np.average(all_fold_best_scores, weights=all_fold_sizes)

    # 2. Standard Deviation
    # For std, a simple np.std is usually fine to just show "stability," 
    # but you can stick to the simple calculation for this.
    std_cv_score = np.std(all_fold_best_scores)
    print('\n' + '='*70)
    print('         FINAL CROSS-VALIDATION SCORE')
    print('='*70)
    print(f'Public LB Score: 0.58')
    # Notice we use the weighted score here
    print(f'Local CV Score: {weighted_cv_score:.4f} ± {std_cv_score:.4f}')
    print('\nIndividual Fold Scores:')
    for i, (score, size) in enumerate(zip(all_fold_best_scores, all_fold_sizes)):
        print(f'  Fold {i+1} (n={size}): {score:.4f}')

Loading data...
357 training images

   FOLD 1/5   |   281 train / 76 val
Building model...
vit_small_patch16_dinov3 parameters: 21586944
Pretrained weights loaded (CPU)
Freezing backbone parameters.


train:   0%|          | 0/281 [00:00<?, ?it/s]/home/teo/kaggle/csiro/.venv/lib/python3.10/site-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 01 | TrainLoss 1840.63338 | ValLoss 2266.63144 | ValR² -1.1852 (BEST)
SAVED (R²: -1.1852)


Epoch 02 | TrainLoss 1501.27470 | ValLoss 1207.30953 | ValR² -0.1641 (BEST)
SAVED (R²: -0.1641)


Epoch 03 | TrainLoss 575.34586 | ValLoss 701.16833 | ValR² 0.3249 (BEST)
SAVED (R²: 0.3249)


...

Epoch 31 | TrainLoss 144.23398 | ValLoss 149.55231 | ValR² 0.7583 (BEST)
SAVED (R²: 0.7583)


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


EARLY STOP (no improvement in 15 epochs)


best_r2,▁▄▇▇▇███████████████████████████████████
train_loss,█▆▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_r2,▁▇▇▇▇▆▇█▇█▇███▇█▆█▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
best_r2,0.75834
train_loss,132.59538
val_loss,181.70644
val_r2,0.70539



         FINAL CROSS-VALIDATION SCORE
Public LB Score: 0.58
Local CV Score: 0.7620 ± 0.0400

Individual Fold Scores:
  Fold 1 (n=76): 0.8364
  Fold 2 (n=74): 0.7457
  Fold 3 (n=90): 0.7420
  Fold 4 (n=57): 0.7195
  Fold 5 (n=60): 0.7583


In [2]:
import timm

# This lists all DINOv3 variants available in your version
models = timm.list_models('*dinov3*')
for m in models:
    print(m)

vit_7b_patch16_dinov3
vit_base_patch16_dinov3
vit_base_patch16_dinov3_qkvb
vit_huge_plus_patch16_dinov3
vit_huge_plus_patch16_dinov3_qkvb
vit_large_patch16_dinov3
vit_large_patch16_dinov3_qkvb
vit_small_patch16_dinov3
vit_small_patch16_dinov3_qkvb
vit_small_plus_patch16_dinov3
vit_small_plus_patch16_dinov3_qkvb
